# Glacier Monitoring
**NB 13** | Aletsch Glacier, Swiss Alps | Sentinel-2 | NDSI + ChangeFormer

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pathlib, json
import numpy as np
import matplotlib.pyplot as plt
import pygeovision as pgv
from pygeovision.models import get_model
from pygeovision.inference.tiled import TiledInference

client = pgv.PyGeoVision()
print(client)

from pygeovision.models.change_detection.changeformer import ChangeDetection


In [ ]:
BBOX        = (8.00, 46.35, 8.35, 46.60)   # Aletsch Glacier, Valais Switzerland
DATE_2000   = ('2015-08-01', '2015-09-30')  # Historical baseline
DATE_NOW    = ('2024-08-01', '2024-09-30')  # Current year
PROVIDERS   = ['planetary_computer']
BANDS       = ['B03','B08','B11']           # Green, NIR, SWIR1 for NDSI + snowline
DATA_DIR    = pathlib.Path('./results/nb13')
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
r2015 = client.search(bbox=BBOX, date_range=DATE_2000, providers=PROVIDERS, cloud_cover_max=15)
r2024 = client.search(bbox=BBOX, date_range=DATE_NOW,  providers=PROVIDERS, cloud_cover_max=15)
print(f"2015: {len(r2015)} | 2024: {len(r2024)}")
d2015 = client.download(r2015[:1], output_dir=str(DATA_DIR/'2015'), bands=BANDS, post_process=['reproject:EPSG:32632','cog'])
d2024 = client.download(r2024[:1], output_dir=str(DATA_DIR/'2024'), bands=BANDS, post_process=['reproject:EPSG:32632','cog'])
s2015 = d2015[0].path if d2015 and d2015[0].success else None
s2024 = d2024[0].path if d2024 and d2024[0].success else None


In [ ]:
R2015 = DATA_DIR / '2015_ready.tif'; R2024 = DATA_DIR / '2024_ready.tif'
for scene, out, label in [(s2015, R2015,'2015'), (s2024, R2024,'2024')]:
    if scene:
        rep = client.validator.validate(str(scene), required_bands=len(BANDS))
        r   = client.prepare_for_ai(str(scene), stack_bands=BANDS, bbox=BBOX,
            normalise='scale_factor', scale_factor=10000.0,
            model_type='segmentation', output_path=str(out))
        print(f"{label}: {r['shape']}  validated")


In [ ]:
# NDSI glacier extent — requires Green and SWIR1 bands (band positions 1,3 in our stack)
if R2015.exists() and R2024.exists():
    idx_dir = DATA_DIR/'indices'; idx_dir.mkdir(exist_ok=True)
    ndsi_2015 = client.indices.ndsi(str(R2015), green_band=1, swir1_band=3,
        output_path=str(idx_dir/'ndsi_2015.tif'))
    ndsi_2024 = client.indices.ndsi(str(R2024), green_band=1, swir1_band=3,
        output_path=str(idx_dir/'ndsi_2024.tif'))
    import rasterio, numpy as np
    for year, path in [('2015', ndsi_2015), ('2024', ndsi_2024)]:
        with rasterio.open(path) as src:
            a    = src.read(1)
            prof = src.profile.copy()
        glacier_pct = (a > 0.40).mean() * 100
        print(f"NDSI {year}: glacier/snow cover = {glacier_pct:.1f}% (NDSI > 0.40)")
    # Compute retreat
    with rasterio.open(ndsi_2015) as s1, rasterio.open(ndsi_2024) as s2:
        g2015 = s1.read(1) > 0.40
        g2024 = s2.read(1) > 0.40
        res_m = abs(s1.transform.a)
    lost_pixels = (g2015 & ~g2024).sum()
    lost_ha     = lost_pixels * res_m**2 / 10000
    print(f"Glacier retreat 2015→2024: {lost_ha:.1f} ha lost")


In [ ]:
# ChangeFormer to detect glacier retreat areas
RETREAT_MAP = DATA_DIR / 'glacier_retreat.tif'
PREDICTION  = RETREAT_MAP
if R2015.exists() and R2024.exists():
    cd = ChangeDetection(model_variant='changeformer', in_channels=len(BANDS))
    cd.build()
    res = cd.detect(str(R2015), str(R2024), output_path=str(RETREAT_MAP))
    print(f"Changed area: {res['change_pct']:.2f}%")


In [ ]:
SIEVED  = DATA_DIR / 'sieved.tif'
VECTOR  = DATA_DIR / 'output.geojson'
COG_OUT = DATA_DIR / 'output_cog.tif'

if PREDICTION.exists():
    client.postprocess.sieve_filter(str(PREDICTION), min_pixels=10, output_path=str(SIEVED))
    client.postprocess.vectorise(str(SIEVED), str(VECTOR),
        target_class=1, min_area_m2=100.0, simplify_tolerance=1.0)
    stats = client.postprocess.class_statistics(str(SIEVED))
    client.postprocess.to_cog(str(SIEVED), str(COG_OUT))
    with open(VECTOR) as f: gj = json.load(f)
    print(f"Polygons detected : {len(gj['features'])}")
    for cls, info in sorted(stats.items()):
        print(f"  Class {cls}: {info['pixels']:>8,} px | {info['area_ha']:>8.1f} ha | {info['pct']:>6.1f}%")
    print(f"COG saved: {COG_OUT}")


**NB13 — Glacier Monitoring** complete.
- Study area: Aletsch Swiss Alps
- Sensor: Sentinel-2 bi-temporal
- Model: NDSI + ChangeFormer
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG